# Debug sparse 3DCal masks

This notebook tests the baseline patch-space masking. Targets are sampled only from active patches, context and target are disjoint, and empty target regions are rejected.

In [ ]:
from pathlib import Path
import sys
import numpy as np

PROJECT = Path('/home/fcufino/FabioFaserDL/projects/Nu-JEPA')
sys.path.insert(0, str(PROJECT))

from src.data.faser_npz_dataset import list_npz_files
from src.data.sparse_preprocessing import extract_reco_hits, hits_to_sparse_event, summarize_active_patches, split_event_by_patch_mask
from src.masking.block3d_masking import Block3DMaskGenerator
from src.visualization.plotly_3dcal import scatter_event, plot_masked_event, mask_sanity_table

DATASET = Path('/scratch/fcufino/events_v8.0_1000')
DETECTOR_SIZE = (48, 48, 200)
PATCH_SIZE = (12, 12, 10)
GRID_SIZE = tuple((d + p - 1) // p for d, p in zip(DETECTOR_SIZE, PATCH_SIZE))
files = list_npz_files(str(DATASET))
print(f'Found {len(files)} npz files under {DATASET}')
if not files:
    raise FileNotFoundError(f'No .npz files found under {DATASET}')

In [ ]:
path = files[0]
with np.load(path, allow_pickle=True) as npz:
    hits = extract_reco_hits(npz)
event = hits_to_sparse_event(hits, detector_size=DETECTOR_SIZE, coord_mode='discrete', path=path)
patches = summarize_active_patches(event, detector_size=DETECTOR_SIZE, patch_size=PATCH_SIZE)
print('event:', path)
print('active voxels:', event.coords.shape[0])
print('active patches:', patches.patch_ids.numel())
print('grid size:', GRID_SIZE)

In [ ]:
masker = Block3DMaskGenerator(
    grid_size=GRID_SIZE,
    num_target_blocks=4,
    target_block_shape=(1, 1, 2),
    min_target_patches=1,
    max_target_fraction=0.45,
    min_context_patches=2,
    seed=17,
)
mask = masker(patches.patch_coords, patches.patch_ids, patches.energy)
sanity = mask_sanity_table(event, patches, mask)
sanity

In [ ]:
assert sanity['overlap_patches'] == 0
assert sanity['target_patches'] > 0
assert sanity['context_patches'] > 0
assert sanity['context_patches'] + sanity['target_patches'] <= sanity['active_patches']
print('Mask sanity checks passed')

In [ ]:
scatter_event(event, title='Full event').show()

In [ ]:
split = split_event_by_patch_mask(
    event,
    detector_size=DETECTOR_SIZE,
    patch_size=PATCH_SIZE,
    target_patch_ids=mask.target_patch_ids.tolist(),
)
scatter_event(event, title='Context input to context encoder', mask=split['context_voxel_mask']).show()
scatter_event(event, title='Target region encoded by target encoder', mask=split['target_voxel_mask']).show()

In [ ]:
plot_masked_event(event, mask, detector_size=DETECTOR_SIZE, patch_size=PATCH_SIZE).show()